# Embeddings pre-treinados: Word2Vec, FastText e GloVe

Este notebook testa modelos pre-treinados usando tambem o JSON dos artigos STIL 2023 como fonte de palavras e pares para consulta. Ele carrega vetores prontos, compara palavras por similaridade e permite visualizar palavras proximas em 2D.

Observacao: os modelos Word2Vec Google News e FastText Wiki News sao grandes e podem demorar no Colab. Se quiser um teste rapido, carregue primeiro apenas o GloVe.

In [ ]:
%pip install -q gensim scikit-learn pandas matplotlib

In [ ]:
import json
import re
from collections import Counter

import gensim.downloader as api
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

## Carregar o JSON dos artigos

Este notebook usa modelos prontos, mas as palavras consultadas saem do corpus STIL 2023. No Colab, faca upload do arquivo `stil2023_articles.json` ou `stil2023_articles (1).json`.

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    json_path = next(iter(uploaded.keys()))
except Exception:
    json_path = "stil2023_articles (1).json"

with open(json_path, "r", encoding="utf-8") as f:
    artigos = json.load(f)

print(f"Artigos carregados: {len(artigos)}")

In [ ]:
def reparar_mojibake(texto):
    if not isinstance(texto, str):
        return ""
    try:
        return texto.encode("latin1").decode("utf-8")
    except Exception:
        return texto


def extrair_texto_artigo(artigo):
    partes = []
    for campo in ["titulo", "resumo", "keywords", "artigo_completo"]:
        valor = artigo.get(campo, "")
        if isinstance(valor, list):
            valor = " ".join(str(item) for item in valor)
        partes.append(reparar_mojibake(str(valor)))
    return " ".join(partes).lower()


def tokenizar(texto):
    return re.findall(r"[a-zA-Z][a-zA-Z\-]{2,}", texto.lower())


tokens_corpus = []
for artigo in artigos:
    tokens_corpus.extend(tokenizar(extrair_texto_artigo(artigo)))

frequencias_corpus = Counter(tokens_corpus)
vocabulario_corpus = [palavra for palavra, freq in frequencias_corpus.most_common() if freq >= 2]

display(pd.DataFrame(frequencias_corpus.most_common(20), columns=["palavra", "frequencia"]))

## Modelos disponíveis

- `glove-wiki-gigaword-100`: GloVe pré-treinado, menor e mais rápido.
- `word2vec-google-news-300`: Word2Vec pré-treinado, grande.
- `fasttext-wiki-news-subwords-300`: FastText pré-treinado, grande e com suporte a subpalavras.

In [ ]:
MODELOS_PRE_TREINADOS = {
    "glove": "glove-wiki-gigaword-100",
    "word2vec": "word2vec-google-news-300",
    "fasttext": "fasttext-wiki-news-subwords-300",
}

# Para Colab com pouco tempo, use ["glove"].
# Para cumprir a comparação completa, use ["glove", "word2vec", "fasttext"].
modelos_para_carregar = ["glove"]

modelos = {}
for nome, codigo in MODELOS_PRE_TREINADOS.items():
    if nome in modelos_para_carregar:
        print(f"Carregando {nome}: {codigo}")
        modelos[nome] = api.load(codigo)
        print(f"{nome} carregado com {len(modelos[nome].index_to_key):,} palavras.")

In [ ]:
def palavras_similares(modelo, palavra, topn=10):
    if palavra not in modelo:
        return pd.DataFrame({"aviso": [f"A palavra '{palavra}' nao esta no vocabulario do modelo."]})
    return pd.DataFrame(modelo.most_similar(palavra, topn=topn), columns=["palavra", "similaridade"])


def palavras_do_corpus_no_modelo(modelo, limite=30):
    palavras = [palavra for palavra in vocabulario_corpus if palavra in modelo]
    return palavras[:limite]


for nome, modelo in modelos.items():
    palavras_disponiveis = palavras_do_corpus_no_modelo(modelo, limite=30)
    palavra_teste = palavras_disponiveis[0] if palavras_disponiveis else "language"
    print(f"\nModelo: {nome}")
    print(f"Palavra escolhida do corpus: {palavra_teste}")
    display(palavras_similares(modelo, palavra_teste, topn=10))

In [ ]:
def comparar_pares(modelo, pares):
    linhas = []
    for a, b in pares:
        if a in modelo and b in modelo:
            score = modelo.similarity(a, b)
            linhas.append({"palavra_1": a, "palavra_2": b, "similaridade": score})
        else:
            linhas.append({"palavra_1": a, "palavra_2": b, "similaridade": None})
    return pd.DataFrame(linhas)


for nome, modelo in modelos.items():
    palavras_disponiveis = palavras_do_corpus_no_modelo(modelo, limite=12)
    pares = list(zip(palavras_disponiveis[::2], palavras_disponiveis[1::2]))
    if not pares:
        pares = [("language", "linguistics"), ("text", "corpus")]
    print(f"\nModelo: {nome}")
    display(comparar_pares(modelo, pares))

In [ ]:
def plotar_vizinhos(modelo, palavra, topn=20, titulo=None):
    if palavra not in modelo:
        print(f"A palavra '{palavra}' nao esta no vocabulario.")
        return

    vizinhos = [palavra] + [w for w, _ in modelo.most_similar(palavra, topn=topn)]
    vetores = [modelo[w] for w in vizinhos]
    coords = PCA(n_components=2, random_state=42).fit_transform(vetores)

    plt.figure(figsize=(10, 7))
    plt.scatter(coords[:, 0], coords[:, 1])
    for i, token in enumerate(vizinhos):
        plt.annotate(token, (coords[i, 0], coords[i, 1]))
    plt.title(titulo or f"Vizinhos de '{palavra}'")
    plt.grid(alpha=0.2)
    plt.show()


for nome, modelo in modelos.items():
    palavras_disponiveis = palavras_do_corpus_no_modelo(modelo, limite=1)
    palavra_teste = palavras_disponiveis[0] if palavras_disponiveis else "language"
    plotar_vizinhos(modelo, palavra_teste, topn=20, titulo=f"{nome}: palavras proximas de {palavra_teste}")